# Prelaboratorio: spaCy en accion sobre una noticia

**Duracion estimada:** 25 minutos

## Proposito
Este notebook funciona como puente entre los materiales introductorios y el laboratorio integrador. Vas a repasar el pipeline completo de `spaCy` sobre una noticia breve y a practicar una lectura critica de los resultados.

## Prerrequisitos
- Haber recorrido `000_intro_pln_y_spacy.ipynb`, `001_spacy_fundamentos.ipynb` y `002_spacy_pipeline_y_apoyo.ipynb`.
- Manejar listas, diccionarios y comprensiones basicas en Python.


## Modalidad sugerida: trabajo con IA como copiloto cognitivo

En esta materia la IA forma parte del dispositivo pedagogico. Podes usar un asistente de IA para:
- anticipar que entidades o verbos podrian aparecer;
- proponer una estrategia de analisis;
- explicarte una salida que no entendiste;
- auditar una interpretacion que te parezca dudosa.

**Importante:** la IA procesa patrones, pero no decide por vos. La lectura final de los resultados tiene que ser humana, situada y justificada.


In [ ]:
!pip install spacy pandas matplotlib wordcloud -q
!python -m spacy download es_core_news_lg -q


In [ ]:
import spacy
import pandas as pd
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from spacy import displacy

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_colwidth', None)

nlp = spacy.load("es_core_news_lg")
print("Entorno configurado correctamente.")


## 1. Repaso rapido: que herramientas ya conoces

Antes de mirar la noticia, recupera estas ideas:
- tokenizacion: como se segmenta el texto;
- lematizacion: como se lleva cada palabra a una forma base;
- POS: como se etiqueta la funcion gramatical;
- NER: como se detectan entidades del mundo real;
- dependencias: como se vinculan las palabras entre si.


## 2. Demostracion: un pipeline completo con spaCy

Vamos a trabajar con un texto breve de estilo periodistico. Antes de ejecutarlo, podes pedirle a un asistente de IA que anticipe:
- que personas y organizaciones cree que van a aparecer;
- que verbos podrian ser centrales;
- que errores posibles podria cometer el modelo.


In [ ]:
print("Cargando modelo de spaCy para espanol...")
print(f"Componentes del pipeline: {nlp.pipe_names}\n")

texto_noticia = (
    "El presidente Javier Milei anuncio ayer en Casa Rosada un nuevo plan de inversiones por "
    "500 millones de dolares. Microsoft Argentina y Mercado Libre seran las primeras empresas "
    "en participar. Maria Gonzalez, ministra de Tecnologia, destaco que el proyecto generara "
    "3.000 empleos en Buenos Aires y Cordoba durante 2025."
)

print("--- Texto a analizar ---")
print(texto_noticia)

doc = nlp(texto_noticia)
print(f"\nTexto procesado: se encontraron {len(doc)} tokens.")


### 2.1 Tokenizacion detallada

En esta salida conviene observar no solo el token, sino tambien el lema, la categoria gramatical y si el modelo lo considera una stopword.


In [ ]:
print("--- Analisis de los primeros 12 tokens ---")
print(f"{'Token':<18} {'Lema':<18} {'POS':<10} {'Stopword?':<12} {'Puntuacion?'}")
print("-" * 75)

for token in doc[:12]:
    print(
        f"{token.text:<18} {token.lemma_:<18} {token.pos_:<10} "
        f"{'Si' if token.is_stop else 'No':<12} {'Si' if token.is_punct else 'No'}"
    )


### 2.2 Reconocimiento de entidades nombradas

El valor pedagogico de esta parte no esta solo en detectar entidades, sino tambien en discutirlas. Si alguna clasificacion te resulta extrana, no la tomes como un fracaso del ejercicio: tomala como una oportunidad de analisis.


In [ ]:
entidades_data = []
for ent in doc.ents:
    entidades_data.append({
        'Texto': ent.text,
        'Tipo': ent.label_,
        'Explicacion': spacy.explain(ent.label_),
    })

df_entidades = pd.DataFrame(entidades_data)

print("--- Entidades nombradas detectadas ---")
if not df_entidades.empty:
    print(df_entidades.to_string(index=False))
else:
    print("No se encontraron entidades en el texto.")


In [ ]:
displacy.render(doc, style='ent', jupyter=True)


**Pausa critica**

Elegi una entidad de la tabla y responde:
- La clasificacion te parece adecuada?
- Que evidencia del texto la sostiene?
- Que podria haber confundido al modelo?


### 2.3 Acciones y estructura sintactica

Ahora vamos a mirar verbos y dependencias para entender mejor que acciones aparecen y como se organiza la primera oracion.


In [ ]:
verbos = [token.lemma_.lower() for token in doc if token.pos_ == "VERB" and not token.is_stop]
verbos_freq = Counter(verbos)

print("--- Verbos principales ---")
if verbos:
    print(verbos_freq.most_common(5))
else:
    print("No se encontraron verbos relevantes.")

primera_oracion = list(doc.sents)[0]
print("\n--- Primera oracion ---")
print(primera_oracion)


In [ ]:
displacy.render(primera_oracion, style='dep', jupyter=True, options={'distance': 120, 'compact': True})


## 3. Extraccion de palabras clave

En esta parte vamos a sintetizar el texto mediante palabras clave. La idea no es reemplazar la lectura humana, sino construir una primera capa de interpretacion.


In [ ]:
palabras_clave = [
    token.lemma_.lower()
    for token in doc
    if token.is_alpha and not token.is_stop and len(token.text) > 2 and token.pos_ in ['NOUN', 'VERB', 'PROPN']
]

frecuencia_palabras = Counter(palabras_clave)

print("--- Top 5 palabras clave ---")
for palabra, freq in frecuencia_palabras.most_common(5):
    print(f"- {palabra}: {freq}")


### 3.1 Visualizacion rapida

La nube de palabras te ofrece una vista panoramica. Mirala como apoyo, no como conclusion final.


In [ ]:
if frecuencia_palabras:
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis',
        max_words=50,
    ).generate_from_frequencies(frecuencia_palabras)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.set_title('Palabras clave de la noticia', fontsize=16)
    ax.axis('off')
    plt.show()
else:
    print("No hay suficientes palabras para generar la nube.")


## 4. Adelanto del laboratorio integrador

En el proximo notebook vas a construir un analizador de noticias mas completo. La diferencia es que ya no vas a trabajar con un ejemplo resuelto, sino con un laboratorio con huecos para completar.

**Modalidad de trabajo del laboratorio:** pair programming con IA.

En esta catedra, eso significa que la diada de trabajo esta formada por vos y un asistente de IA. La IA puede proponer, explicar, depurar y contrastar, pero no reemplaza tu criterio ni tu responsabilidad sobre el resultado.


## 5. Antes de pasar al laboratorio

Comproba si podes hacer estas tres cosas:
1. explicar que informacion nueva agrega `spaCy` a un texto;
2. identificar una salida discutible del modelo y justificar por que;
3. usar un asistente de IA para formular una hipotesis y despues contrastarla con la salida real.

**Microbitacora sugerida**
- Que le pediste a la IA?
- Que parte de la respuesta te resulto util?
- Que corregiste o descartaste?
- Que aprendiste al comparar la prediccion con la ejecucion del notebook?
